# 24.2 设计搜索排序系统 / Design a Search Ranking System (Google / Amazon scale)

**中文**:搜索和推荐(24.1)是"孪生兄弟",共享**多阶段漏斗**骨架,但搜索多了一个决定性的东西——**用户输入的查询(query)**。这让搜索既更简单(有明确的意图信号)又更难(要理解查询、匹配查询与文档的相关性)。设计"Google 搜索"或"Amazon 商品搜索"是最经典的系统设计题之一。它的核心技术有两块:①**检索**——怎么从十亿网页里快速找到相关的?答案是**混合检索:词汇匹配(BM25)+ 语义匹配(向量)**,两者互补;②**排序**——怎么把召回的文档按相关性排好?答案是**排序学习(LTR,接 20.12)**。本节从零实现 BM25 和语义检索,亲眼看到"为什么词汇和语义必须混合",再讲清搜索系统设计的完整框架。
**English**: Search and recommendation (24.1) are "twins" sharing the **multi-stage funnel** skeleton, but search adds one decisive thing — **the user's query**. This makes search both simpler (a clear intent signal) and harder (understand the query, match query-document relevance). Designing "Google search" or "Amazon product search" is one of the most classic system-design questions. Its core technology has two parts: ① **retrieval** — how to quickly find relevant results from billions of pages? The answer is **hybrid retrieval: lexical matching (BM25) + semantic matching (vectors)**, which complement each other; ② **ranking** — how to order the retrieved docs by relevance? The answer is **learning to rank (LTR, per 20.12)**. This section implements BM25 and semantic retrieval from scratch, seeing why lexical and semantic must be hybrid, then clarifies the complete search-system-design framework.

---

**中文**:**搜索系统的漏斗(和推荐同构,但每层有搜索特色)**:
**English**: **The search funnel (isomorphic to recommendation, but each layer has search specifics)**:
- **中文**:**① 查询理解(query understanding)**:纠错(拼写)、分词、意图识别、实体识别、查询改写/扩展(同义词)。这是搜索独有的前置步骤——理解"用户到底想要什么"。
  **① Query understanding**: spell correction, tokenization, intent classification, entity recognition, query rewriting/expansion (synonyms). This is search's unique front step — understand "what the user really wants."
- **中文**:**② 检索(retrieval)—— 混合是关键**:**词汇检索(BM25 + 倒排索引)** 擅长精确关键词匹配(型号、人名、专有名词),但对同义改写无能为力(查"budget airline"匹配不到"cheap flights");**语义检索(dense embedding + ANN)** 擅长理解意思(同义词、改写),但对精确关键词/罕见词较弱。**两者混合(hybrid)取长补短**,是现代搜索的标配。
  **② Retrieval — hybrid is key**: **lexical retrieval (BM25 + inverted index)** excels at exact keyword matching (model numbers, names, proper nouns) but fails on paraphrases (query "budget airline" won't match "cheap flights"); **semantic retrieval (dense embedding + ANN)** excels at understanding meaning (synonyms, rewrites) but is weaker on exact/rare keywords. **Hybridizing the two** complements their strengths and is standard in modern search.
- **中文**:**③ 排序(ranking)—— LTR**:对召回的文档,用**排序学习模型(接 20.12)** 按相关性精排,特征包括查询-文档相关性(BM25 分、语义相似度)、文档质量(权威性、点击率)、用户/上下文。目标指标:**NDCG**(位置折扣,头部最重要)。
  **③ Ranking — LTR**: rank the retrieved docs by relevance with a **learning-to-rank model (per 20.12)**, with features like query-doc relevance (BM25 score, semantic similarity), doc quality (authority, click-through), user/context. Target metric: **NDCG** (positional discount, top matters most).
- **中文**:**④ 重排**:多样性(别一屏全是一个来源)、去重、业务规则(广告、置顶)。
  **④ Re-ranking**: diversity (not one source filling the screen), dedup, business rules (ads, pinning).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 搜索系统设计, 高频）**
> **中文**:**搜索 vs 推荐**:都用多阶段漏斗, 但搜索有**明确查询**(意图信号强)→多了**查询理解**(纠错/分词/意图/实体/改写)。**检索(核心)=混合**:①**词汇 BM25 + 倒排索引**(精确关键词/型号/人名, 但不懂同义)②**语义 dense embedding + ANN**(懂意思/同义改写, 但弱于精确词)→**hybrid 两者融合**(现代标配, 如 BM25+向量召回取并集/加权)。**排序=LTR**(20.12): pointwise/pairwise(RankNet)/listwise(LambdaMART), 特征=查询-文档相关性(BM25/语义分)+文档质量(PageRank/权威/CTR)+用户上下文; 指标 **NDCG/MRR**。**关键难题**:①**相关性标注**(人工评分+点击日志, 稀疏主观)②**位置偏差**(点击≠相关, 用无偏 LTR/IPW 接 20.12/19)③**查询理解**(长尾/歧义/意图)④**新鲜度**(新闻类要新)⑤**难负样本**(训练检索要挖 hard negatives)。**评估**:离线 NDCG + 在线 A/B(点击、无点击率、满意度)。**规模**:倒排索引分片、向量 ANN(Faiss/ScaNN)、缓存热门查询。面试金句:*"搜索系统:查询理解(纠错/意图/改写)→混合检索(BM25 词汇精确匹配 + 向量语义匹配, 两者互补, 缺一漏召回)→LTR 精排(查询-文档相关性+文档质量特征, 优化 NDCG)→重排多样性; 关键难点是相关性标注稀疏、点击有位置偏差(无偏 LTR 纠偏)、查询理解长尾; 离线 NDCG 选模型, 在线 A/B 定胜负。"*
> **English**: **Search vs recommendation**: both use the multi-stage funnel, but search has an **explicit query** (strong intent signal) → adds **query understanding** (spell/tokenize/intent/entity/rewrite). **Retrieval (core) = hybrid**: ① **lexical BM25 + inverted index** (exact keywords/model numbers/names, but no synonym understanding) ② **semantic dense embedding + ANN** (understands meaning/paraphrase, but weaker on exact words) → **hybrid fuses both** (modern standard, e.g. union/weighting of BM25 + vector retrieval). **Ranking = LTR** (20.12): pointwise/pairwise (RankNet)/listwise (LambdaMART), features = query-doc relevance (BM25/semantic score) + doc quality (PageRank/authority/CTR) + user context; metrics **NDCG/MRR**. **Key challenges**: ① **relevance labeling** (human ratings + click logs, sparse & subjective) ② **position bias** (clicks ≠ relevance, use unbiased LTR/IPW per 20.12/19) ③ **query understanding** (long-tail/ambiguous/intent) ④ **freshness** (news needs recency) ⑤ **hard negatives** (mining hard negatives for retrieval training). **Evaluation**: offline NDCG + online A/B (clicks, no-click rate, satisfaction). **Scale**: sharded inverted index, vector ANN (Faiss/ScaNN), cache popular queries. Interview line: *"Search system: query understanding (spell/intent/rewrite) → hybrid retrieval (BM25 lexical exact matching + vector semantic matching, complementary, either alone misses recall) → LTR ranking (query-doc relevance + doc quality features, optimizing NDCG) → re-ranking for diversity; key challenges are sparse relevance labeling, position bias in clicks (unbiased LTR corrects it), long-tail query understanding; offline NDCG picks models, online A/B decides."*


In [ ]:

# ============================================================
# 从零实现混合检索:BM25(词汇)+ 语义(向量) / hybrid retrieval from scratch: BM25 (lexical) + semantic (vector)
# 中文:搜索检索的核心。BM25 靠原词重叠(擅长精确关键词, 不懂同义); 语义靠向量相似(懂同义, 弱于精确词)。
#      演示两者各自的盲区, 以及混合如何取长补短。
# English: the core of search retrieval. BM25 relies on word overlap (great at exact keywords, no synonyms); semantic
#      relies on vector similarity (understands synonyms, weaker on exact words). Show each's blind spot and how hybrid helps.
# ============================================================
import numpy as np, math
from collections import Counter
corpus=["cheap flights to new york city","affordable airfare to nyc","best pizza restaurants in new york",
        "how to cook italian pasta at home","new york city travel guide budget","flight ticket prices manhattan"]
docs=[d.lower().split() for d in corpus]
N=len(docs); avgdl=np.mean([len(d) for d in docs]); k1,b=1.5,0.75
df=Counter()
for d in docs:
    for w in set(d): df[w]+=1                                    # 文档频率(倒排索引统计)/ document frequency
def bm25(query):                                                # BM25 词汇打分 / BM25 lexical scoring
    q=query.lower().split(); scores=np.zeros(N)
    for i,d in enumerate(docs):
        tf=Counter(d); dl=len(d)
        for w in q:
            if w in tf:
                idf=math.log((N-df[w]+0.5)/(df[w]+0.5)+1)        # 稀有词权重更高 / rare words weigh more
                scores[i]+=idf*tf[w]*(k1+1)/(tf[w]+k1*(1-b+b*dl/avgdl))
    return scores
# 玩具语义向量:让同义词共享向量成分(模拟训练好的 embedding)/ toy embeddings: synonyms share components
np.random.seed(0); vocab={w for d in docs for w in d}; base={w:np.random.randn(16) for w in vocab}
for grp in [["cheap","affordable","budget"],["flights","airfare","flight","ticket","airline"],["nyc","york","manhattan","city"]]:
    v=np.random.randn(16)
    for w in grp:
        if w in base: base[w]=0.2*base[w]+v                      # 同义词→相似向量 / synonyms → similar vectors
emb=lambda toks: np.mean([base[w] for w in toks if w in base] or [np.zeros(16)],0)
doc_emb=np.array([emb(d) for d in docs])
def semantic(query):                                            # 语义相似度打分 / semantic similarity scoring
    q=emb(query.lower().split()); return doc_emb@q/(np.linalg.norm(doc_emb,axis=1)*np.linalg.norm(q)+1e-9)
def top3(scores): return [corpus[i] for i in np.argsort(-scores)[:3]]

query="budget airline tickets manhattan"                        # 'budget airline' 是同义词, 没有原词命中 / synonyms, no exact hit
print(f"查询 query: '{query}'\n")
print("① BM25(词汇匹配)top3 —— 只能匹配含原词的:"); [print("   ·",d) for d in top3(bm25(query))]
print("\n② 语义(向量)top3 —— 理解 budget≈cheap, airline≈flight:"); [print("   ·",d) for d in top3(semantic(query))]
bs,ss=bm25(query),semantic(query)
hybrid=0.5*bs/(bs.max()+1e-9)+0.5*(ss-ss.min())/(ss.max()-ss.min()+1e-9)   # 归一化后加权融合 / normalized weighted fusion
print("\n③ 混合 hybrid top3 —— 词汇+语义取长补短:"); [print("   ·",d) for d in top3(hybrid)]
print("\n→ BM25 抓精确关键词但漏同义改写; 语义懂意思但弱于精确词; 混合检索两者兼得, 是现代搜索标配")


In [ ]:

# ============================================================
# 可视化:搜索漏斗 + 词汇/语义/混合的召回对比 / search funnel + lexical/semantic/hybrid recall
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 搜索漏斗 / search funnel
ax[0].axis("off"); ax[0].set_title("搜索系统漏斗",fontsize=12,weight="bold")
steps=[("查询 query","#9467BD"),("① 查询理解(纠错/意图/改写)","#DD8452"),
       ("② 混合检索 BM25+语义(十亿→千)","#4C72B0"),("③ LTR 精排(NDCG, 千→十)","#55A868"),("④ 重排(多样性/去重)","#8172B3")]
for i,(s,c) in enumerate(steps):
    ax[0].add_patch(plt.Rectangle((0.1,0.8-i*0.17),0.8,0.13,fc=c,alpha=0.3,ec=c,transform=ax[0].transAxes))
    ax[0].text(0.5,0.865-i*0.17,s,ha="center",va="center",fontsize=9,transform=ax[0].transAxes)
    if i<4: ax[0].annotate("",xy=(0.5,0.8-i*0.17),xytext=(0.5,0.82-i*0.17),arrowprops=dict(arrowstyle="->"),transform=ax[0].transAxes)
# ② 三种检索找到相关文档的能力(示意)/ retrieval capability
methods=["仅 BM25\n(词汇)","仅语义\n(向量)","混合\nhybrid"]
exact=[95,60,95]   # 精确关键词召回 / exact-keyword recall
synonym=[30,90,90] # 同义改写召回 / synonym-paraphrase recall
x=np.arange(3); w=0.35
ax[1].bar(x-w/2,exact,w,label="精确关键词召回",color="#4C72B0")
ax[1].bar(x+w/2,synonym,w,label="同义改写召回",color="#DD8452")
ax[1].set_xticks(x); ax[1].set_xticklabels(methods); ax[1].set_ylabel("召回率 %(示意)")
ax[1].set_title("混合检索:精确+同义两种召回都强"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/sd02_viz.png",dpi=80); plt.show()
print("左:搜索漏斗(查询理解→混合检索→LTR精排→重排); 右:BM25 强于精确词、语义强于同义, 混合两者兼得")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **搜索的独特性来自"查询",它既是礼物也是难题**:相比推荐(要猜用户想要什么),搜索有一个明确的意图信号——用户主动打出的查询。但这份礼物也带来搜索独有的难题:**查询理解**。真实查询充满拼写错误、歧义("apple"是水果还是公司?)、口语化、长尾(大量查询从没见过)。所以搜索系统在检索之前必须先**理解查询**(纠错、意图分类、实体识别、查询改写),这是推荐系统没有的一层。能主动提到"查询理解"这一步,是搜索系统设计的加分项。
2. **混合检索是搜索的现代标准答案,因为词汇和语义各有致命盲区**:我们的 demo 清楚地展示了这一点。**BM25(词汇匹配)** 靠原词重叠,它对精确关键词(型号 "iPhone 15 Pro"、人名、专有名词)极准,但遇到同义改写就抓瞎——查 "budget airline" 匹配不到 "cheap flights",因为没有一个词相同。**语义检索(向量)** 反过来:它理解 "budget≈cheap、airline≈flight",能召回同义改写,但对精确的罕见词/型号/代码反而不如 BM25 可靠(向量会把 "iPhone 15" 和 "iPhone 14" 拉得很近)。**两者的盲区恰好互补**,所以现代搜索都用**混合检索**——BM25 保证精确匹配不丢,语义保证意思理解到位。只用其中一个,都会在另一半查询上悄悄漏召回。这是搜索系统设计里最能体现技术深度的一点。
3. **诚实的难点:搜索最难的是"相关性到底怎么定义和衡量",以及点击的偏差**。①**相关性是主观且难标注的**:什么叫"相关"?人工标注昂贵、稀疏、标注者之间还不一致;于是大量依赖**点击日志**作为相关性的代理——但这引出了搜索(和推荐)共同的头号陷阱:**位置偏差**。②**点击≠相关**:用户更爱点排在前面的结果,哪怕它没那么相关;所以"这个结果被点得多"可能只是因为"它排在前面",而非"它更好"。直接用点击训练排序模型,会让模型学到"位置"而非"相关性",形成自我强化的偏差(接 20.12、19 因果)。解法是**无偏 LTR**(用 IPW 给点击按位置加权纠偏)和精心设计的实验。③**难负样本(hard negatives)**:训练语义检索时,随机负样本太容易,模型学不到细粒度区分;要挖**难负样本**(和查询相似但不相关的文档)。④**评估的层次**:离线 NDCG 只是代理,最终看**在线 A/B**(点击率、无点击率、query 重构率、满意度)——和推荐一样,离线好≠在线好。**结论:设计搜索系统在推荐漏斗基础上加了查询理解, 核心是混合检索(BM25 精确+语义同义, 互补缺一漏召回)和 LTR 精排(优化 NDCG); 但真正的深水区是相关性的定义与标注、点击的位置偏差(需无偏 LTR)、难负样本挖掘、以及离线只是代理最终看在线 A/B——搜索的功力在数据与相关性判断, 不止在检索算法。**

**English**:
1. **Search's uniqueness comes from "the query," both a gift and a problem**: compared to recommendation (which guesses what the user wants), search has an explicit intent signal — the query the user actively typed. But this gift brings search's unique problem: **query understanding**. Real queries are full of spelling errors, ambiguity ("apple" — fruit or company?), colloquialisms, and long-tail (many queries never seen). So a search system must **understand the query** (spell correction, intent classification, entity recognition, query rewriting) before retrieval — a layer recommendation lacks. Proactively mentioning "query understanding" is a plus in search-system design.
2. **Hybrid retrieval is search's modern standard answer, because lexical and semantic each have fatal blind spots**: our demo shows this clearly. **BM25 (lexical matching)** relies on word overlap — extremely precise on exact keywords (model number "iPhone 15 Pro", names, proper nouns) but blind to paraphrases — query "budget airline" won't match "cheap flights" because no word is shared. **Semantic retrieval (vectors)** is the opposite: it understands "budget≈cheap, airline≈flight" and recalls paraphrases, but is less reliable than BM25 on exact rare words/model numbers/codes (vectors pull "iPhone 15" and "iPhone 14" close). **Their blind spots exactly complement**, so modern search uses **hybrid retrieval** — BM25 ensures exact matches aren't lost, semantic ensures meaning is captured. Using only one silently misses recall on the other half of queries. This is where search-system design best shows technical depth.
3. **Honest difficulty: search's hardest part is "how relevance is actually defined and measured," and click bias**. ① **Relevance is subjective and hard to label**: what is "relevant"? Human labeling is expensive, sparse, and inconsistent between labelers; so search leans heavily on **click logs** as a relevance proxy — but this raises the #1 trap shared by search (and recommendation): **position bias**. ② **Clicks ≠ relevance**: users prefer clicking higher-ranked results even if less relevant; so "this result got many clicks" may only be because "it ranked higher," not "it's better." Training a ranking model directly on clicks makes it learn "position" not "relevance," a self-reinforcing bias (per 20.12, 19 causal). The fix is **unbiased LTR** (IPW to reweight clicks by position) and careful experiments. ③ **Hard negatives**: training semantic retrieval with random negatives is too easy, so the model can't learn fine distinctions; mine **hard negatives** (docs similar to the query but irrelevant). ④ **Layers of evaluation**: offline NDCG is just a proxy; the final judge is **online A/B** (click-through, no-click rate, query-reformulation rate, satisfaction) — like recommendation, offline good ≠ online good. **Conclusion: designing search adds query understanding to the recommendation funnel, with hybrid retrieval (BM25 exact + semantic synonym, complementary, either alone misses recall) and LTR ranking (optimizing NDCG) at the core; but the real deep end is defining and labeling relevance, position bias in clicks (needing unbiased LTR), hard-negative mining, and offline being just a proxy with online A/B the final judge — search's skill is in data and relevance judgment, not just the retrieval algorithm.**

> 💼 **实战视角 / Practical angle**
> **中文**:搜索系统落地:①**查询理解**:拼写纠错、意图分类、实体识别、查询改写/扩展(同义词、上位词);②**混合检索**:BM25(Elasticsearch/OpenSearch 倒排索引)+ 向量检索(Faiss/向量数据库)召回取并集或加权融合(RRF 倒数排名融合);③**LTR 精排**(LambdaMART/深度 LTR, 20.12), 特征=BM25分+语义相似+文档质量(PageRank/权威/CTR)+新鲜度+个性化;④**位置偏差**用无偏 LTR/IPW 纠偏;⑤**难负样本**挖掘训练检索;⑥**评估**离线 NDCG/MRR + 在线 A/B(点击、无点击率、满意度);⑦缓存热门查询、倒排索引分片、向量 ANN 加速。**答题**:先讲查询理解, 强调混合检索(词汇+语义互补), LTR 优化 NDCG, 主动提位置偏差和 A/B。面试金句:*"搜索=查询理解(纠错/意图/改写)+混合检索(BM25 精确关键词+向量语义, 互补, 缺一漏召回)+LTR 精排(查询-文档相关+文档质量, 优化 NDCG)+重排; 关键难点是相关性标注稀疏主观、点击的位置偏差(无偏 LTR 纠)、难负样本、查询长尾; 离线 NDCG 选模型、在线 A/B 定胜负。"*
> **English**: Search system in practice: ① **query understanding**: spell correction, intent classification, entity recognition, query rewriting/expansion (synonyms, hypernyms); ② **hybrid retrieval**: BM25 (Elasticsearch/OpenSearch inverted index) + vector retrieval (Faiss/vector DB) with union or weighted fusion (RRF reciprocal rank fusion); ③ **LTR ranking** (LambdaMART/deep LTR, 20.12), features = BM25 score + semantic similarity + doc quality (PageRank/authority/CTR) + freshness + personalization; ④ **position bias** corrected with unbiased LTR/IPW; ⑤ **hard-negative** mining for retrieval training; ⑥ **evaluation** offline NDCG/MRR + online A/B (clicks, no-click rate, satisfaction); ⑦ cache popular queries, shard the inverted index, accelerate with vector ANN. **Answering**: cover query understanding first, emphasize hybrid retrieval (lexical + semantic complementary), LTR optimizing NDCG, proactively raise position bias and A/B. Interview line: *"Search = query understanding (spell/intent/rewrite) + hybrid retrieval (BM25 exact keywords + vector semantic, complementary, either alone misses recall) + LTR ranking (query-doc relevance + doc quality, optimizing NDCG) + re-ranking; key challenges are sparse subjective relevance labeling, position bias in clicks (unbiased LTR corrects), hard negatives, long-tail queries; offline NDCG picks models, online A/B decides."*

---
### 小结 / Summary
- **中文**:搜索=推荐漏斗+查询(明确意图); 多了查询理解层(纠错/意图/改写)。
- **English**: Search = recommendation funnel + a query (explicit intent); adds a query-understanding layer (spell/intent/rewrite).
- **中文**:核心=混合检索(BM25 词汇精确 + 向量语义同义, 互补缺一漏召回)+ LTR 精排(优化 NDCG)+ 重排。
- **English**: Core = hybrid retrieval (BM25 lexical exact + vector semantic synonym, complementary, either alone misses recall) + LTR ranking (optimizing NDCG) + re-ranking.
- **中文**:难点:相关性标注、点击的位置偏差(无偏 LTR)、难负样本、离线只是代理最终看在线 A/B。
- **English**: Difficulties: relevance labeling, position bias in clicks (unbiased LTR), hard negatives, offline being just a proxy with online A/B the final judge.
